# 08 — ETL: transformaciones y KPIs sobre el datawarehouse en estrella
### TFM · Análisis multidimensional del estrés estudiantil · Entrega 2

> **Cómo ejecutarlo:** menú **Kernel → Restart Kernel and Run All Cells**. No hay que ejecutar nada a mano ni en un orden especial. Si falta algún requisito, la celda 1 se detiene y dice qué script ejecutar. Se puede relanzar las veces que haga falta: no duplica datos.

## Qué hace este notebook
Es un proceso **ETL de MySQL a MySQL**:
- **Extrae** de la estrella de consumo (`vw_hechos_estres`, `vw_dim_dataset`) y de las tablas ya ingestadas por el notebook 07.
- **Transforma** con Python (pandas): integración, agregación, resumen estadístico, pivote con umbrales clínicos y tabla de contingencia.
- **Carga** el resultado en las tablas del datawarehouse (`dwh_*`), que son hechos de la estrella: guardan solo claves hacia las dimensiones conformes (género, rango de edad, nivel educativo, nivel de estrés, dataset, bloque, subreddit) y sus medidas.
- **Calcula 6 KPIs** y los guarda con histórico en `dwh_kpi_resultados`.

| Paso | Transformación | Tipo | Tabla destino |
|---|---|---|---|
| T1 | Registros de las 9 fuentes con señal real | Integración | `dwh_hechos_estres_unificado` |
| T2 | Prevalencia de estrés por bloque × género × edad | Agregación | `dwh_agg_prevalencia` |
| T3 | Señales lingüísticas de Dreaddit por subreddit y nivel | Resumen estadístico | `dwh_nlp_senales` |
| T4 | DASS por género × edad × nivel educativo | Pivote + umbrales clínicos | `dwh_dass_severidad` |
| T5 | Tratamiento según estrés creciente y antecedentes | Contingencia | `dwh_brecha_tratamiento` |

**Requisitos previos (ya ejecutados en tu base):** `modelo_datos_mysql.sql`, `esquema_estrella_powerbi.sql` y el notebook 07. **El datawarehouse (`dwh_estrella.sql`) no hay que ejecutarlo a mano:** si falta alguna de sus tablas, la celda 1 lo crea sola.

## Celda 1 — Configuración, conexión y comprobación de requisitos

In [ ]:
import os, json
import numpy as np
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, text

DB_USER, DB_PASS = "root", ""
DB_HOST, DB_NAME = "localhost", "tfm_estres_estudiantil_2"
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4")
EJECUCION = datetime.now()

def existe(objeto):
    """True si la tabla o vista existe (sirve para ambas)."""
    try:
        with engine.connect() as con:
            con.execute(text(f"SELECT 1 FROM {objeto} LIMIT 1"))
        return True
    except Exception:
        return False

# Todo lo que este notebook lee o escribe debe existir; si no, se dice exactamente que ejecutar.
REQUISITOS = {
    "modelo_datos_mysql.sql + notebook 07 (datos ingestados)": ["b3_dreaddit_post", "b3_depression_student", "b4_dass_respondente", "b4_mental_health"],
    "esquema_estrella_powerbi.sql (vistas de la estrella)": ["vw_hechos_estres", "vw_dim_dataset"],
    "modelo_datos_mysql.sql (dimensiones conformes)": ["dim_dataset", "dim_bloque", "dim_genero", "dim_rango_edad", "dim_nivel_educativo", "dim_nivel_estres", "dim_subreddit"],
}
faltan = {k: [t for t in v if not existe(t)] for k, v in REQUISITOS.items()}
faltan = {k: v for k, v in faltan.items() if v}
if faltan:
    raise RuntimeError("Antes de continuar:\n" + "\n".join(f"  - falta {', '.join(v)}  ->  ejecuta: {k}" for k, v in faltan.items()))
with engine.connect() as con:
    n_estrella = con.execute(text("SELECT COUNT(*) FROM vw_hechos_estres")).scalar()
if n_estrella == 0:
    raise RuntimeError("La estrella no tiene datos: ejecuta antes el notebook 07 (carga de datos).")

# ── Datawarehouse: si falta alguna tabla (p. ej. no se ejecuto dwh_estrella.sql) se crea aqui, con el mismo script ──
DWH_TABLAS = ["dwh_hechos_estres_unificado", "dwh_agg_prevalencia", "dwh_nlp_senales", "dwh_dass_severidad",
              "dwh_brecha_tratamiento", "dwh_kpi_catalogo", "dwh_kpi_resultados", "dwh_modelo_resultados"]
DDL_DWH = """
SET FOREIGN_KEY_CHECKS = 0;
DROP VIEW  IF EXISTS vw_modelo_resultados;
DROP TABLE IF EXISTS dwh_modelo_resultados, dwh_kpi_resultados, dwh_kpi_catalogo,
                     dwh_brecha_tratamiento, dwh_dass_severidad, dwh_nlp_senales,
                     dwh_agg_prevalencia, dwh_hechos_estres_unificado;
SET FOREIGN_KEY_CHECKS = 1;

-- ============================================================================
-- T1  HECHO CENTRAL  (grano: 1 registro de las 9 fuentes con senal real)
-- ============================================================================
CREATE TABLE dwh_hechos_estres_unificado (
  id_hecho            INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_dataset          TINYINT UNSIGNED NOT NULL,
  id_genero           TINYINT UNSIGNED NOT NULL,
  id_rango_edad       TINYINT UNSIGNED NOT NULL,
  id_nivel_educativo  TINYINT UNSIGNED NOT NULL,
  id_nivel_estres     TINYINT UNSIGNED NOT NULL,
  demografia_completa BOOLEAN NOT NULL COMMENT '1 si genero y edad son reportados',
  PRIMARY KEY (id_hecho),
  CONSTRAINT fk_t1_dataset FOREIGN KEY (id_dataset)         REFERENCES dim_dataset(id_dataset),
  CONSTRAINT fk_t1_genero  FOREIGN KEY (id_genero)          REFERENCES dim_genero(id_genero),
  CONSTRAINT fk_t1_edad    FOREIGN KEY (id_rango_edad)      REFERENCES dim_rango_edad(id_rango_edad),
  CONSTRAINT fk_t1_edu     FOREIGN KEY (id_nivel_educativo) REFERENCES dim_nivel_educativo(id_nivel_educativo),
  CONSTRAINT fk_t1_estres  FOREIGN KEY (id_nivel_estres)    REFERENCES dim_nivel_estres(id_nivel_estres)
) ENGINE=InnoDB;

-- ============================================================================
-- T2  HECHO AGREGADO  (grano: bloque x genero x rango de edad)
-- ============================================================================
CREATE TABLE dwh_agg_prevalencia (
  id_agg            INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_bloque         TINYINT UNSIGNED NOT NULL,
  id_genero         TINYINT UNSIGNED NOT NULL,
  id_rango_edad     TINYINT UNSIGNED NOT NULL,
  n_total           INT UNSIGNED NOT NULL,
  n_estres_alto     INT UNSIGNED NOT NULL,
  n_estres_moderado INT UNSIGNED NOT NULL,
  n_estres_bajo     INT UNSIGNED NOT NULL,
  pct_estres_alto   DECIMAL(5,2) NOT NULL,
  PRIMARY KEY (id_agg),
  CONSTRAINT fk_t2_bloque FOREIGN KEY (id_bloque)     REFERENCES dim_bloque(id_bloque),
  CONSTRAINT fk_t2_genero FOREIGN KEY (id_genero)     REFERENCES dim_genero(id_genero),
  CONSTRAINT fk_t2_edad   FOREIGN KEY (id_rango_edad) REFERENCES dim_rango_edad(id_rango_edad)
) ENGINE=InnoDB;

-- ============================================================================
-- T3  HECHO AGREGADO NLP  (grano: subreddit x nivel de estres)
-- ============================================================================
CREATE TABLE dwh_nlp_senales (
  id_senal            INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_subreddit        SMALLINT UNSIGNED NOT NULL,
  id_nivel_estres     TINYINT UNSIGNED NOT NULL,
  n_posts             INT UNSIGNED NOT NULL,
  sentimiento_medio   DECIMAL(7,4) NOT NULL,
  liwc_ansiedad_medio DECIMAL(7,4) NOT NULL,
  liwc_emo_neg_medio  DECIMAL(7,4) NOT NULL,
  liwc_emo_pos_medio  DECIMAL(7,4) NOT NULL,
  longitud_media_palabras DECIMAL(7,2) NOT NULL,
  PRIMARY KEY (id_senal),
  CONSTRAINT fk_t3_subreddit FOREIGN KEY (id_subreddit)    REFERENCES dim_subreddit(id_subreddit),
  CONSTRAINT fk_t3_estres    FOREIGN KEY (id_nivel_estres) REFERENCES dim_nivel_estres(id_nivel_estres)
) ENGINE=InnoDB;

-- ============================================================================
-- T4  HECHO AGREGADO DASS  (grano: genero x rango de edad x nivel educativo)
-- ============================================================================
CREATE TABLE dwh_dass_severidad (
  id_dass_sev        INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_genero          TINYINT UNSIGNED NOT NULL,
  id_rango_edad      TINYINT UNSIGNED NOT NULL,
  id_nivel_educativo TINYINT UNSIGNED NOT NULL,
  n_respondentes     INT UNSIGNED NOT NULL,
  depresion_media    DECIMAL(6,2) NOT NULL,
  ansiedad_media     DECIMAL(6,2) NOT NULL,
  estres_medio       DECIMAL(6,2) NOT NULL,
  pct_estres_severo  DECIMAL(5,2) NOT NULL COMMENT '% con estres >= 40 (baremo DASS)',
  PRIMARY KEY (id_dass_sev),
  CONSTRAINT fk_t4_genero FOREIGN KEY (id_genero)          REFERENCES dim_genero(id_genero),
  CONSTRAINT fk_t4_edad   FOREIGN KEY (id_rango_edad)      REFERENCES dim_rango_edad(id_rango_edad),
  CONSTRAINT fk_t4_edu    FOREIGN KEY (id_nivel_educativo) REFERENCES dim_nivel_educativo(id_nivel_educativo)
) ENGINE=InnoDB;

-- ============================================================================
-- T5  TABLA DE CONTINGENCIA  (grano: estres creciente x antecedentes familiares)
-- ============================================================================
CREATE TABLE dwh_brecha_tratamiento (
  id_brecha               INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_nivel_estres         TINYINT UNSIGNED NOT NULL COMMENT 'Estres creciente: No=Bajo, Maybe=Moderado, Yes=Alto',
  antecedentes_familiares ENUM('Yes','No') NOT NULL COMMENT 'Dimension degenerada',
  n_total                 INT UNSIGNED NOT NULL,
  n_con_tratamiento       INT UNSIGNED NOT NULL,
  pct_tratamiento         DECIMAL(5,2) NOT NULL,
  PRIMARY KEY (id_brecha),
  CONSTRAINT fk_t5_estres FOREIGN KEY (id_nivel_estres) REFERENCES dim_nivel_estres(id_nivel_estres)
) ENGINE=InnoDB;

-- ============================================================================
-- KPIs: catalogo (definicion) + historico de resultados (1 fila por KPI y ejecucion)
-- ============================================================================
CREATE TABLE dwh_kpi_catalogo (
  id_kpi         TINYINT UNSIGNED NOT NULL,
  codigo         VARCHAR(30)  NOT NULL UNIQUE,
  nombre         VARCHAR(120) NOT NULL,
  descripcion    VARCHAR(400) NOT NULL,
  unidad         VARCHAR(20)  NOT NULL,
  fuente_calculo VARCHAR(60)  NOT NULL,
  PRIMARY KEY (id_kpi)
) ENGINE=InnoDB;

INSERT INTO dwh_kpi_catalogo VALUES
 (1,'KPI_ESTRES_ALTO','Tasa de estres Alto global',
    'Porcentaje de registros en nivel de estres Alto sobre el total de las 9 fuentes con senal real.','%','dwh_hechos_estres_unificado'),
 (2,'KPI_BRECHA_GENERO','Brecha de genero en ansiedad (DASS)',
    'Diferencia entre la ansiedad media de mujeres y hombres en la escala DASS-42 (18-25 anos).','puntos','b4_dass_respondente'),
 (3,'KPI_RIESGO_CRITICO','Indice de riesgo critico',
    'Porcentaje de estudiantes con pensamientos suicidas que ademas presentan depresion.','%','b3_depression_student'),
 (4,'KPI_TASA_TRATAMIENTO','Tasa de busqueda de tratamiento',
    'Porcentaje de estudiantes que han buscado tratamiento psicologico.','%','dwh_brecha_tratamiento'),
 (5,'KPI_DISCRIMINANTE_LIWC','Poder discriminante LIWC',
    'Ratio entre la densidad media de palabras de ansiedad (LIWC) de la clase estres y la de no-estres en Dreaddit.','ratio','b3_dreaddit_post'),
 (6,'KPI_COBERTURA_DEMO','Cobertura demografica',
    'Porcentaje de registros con genero y edad reportados sobre el total unificado.','%','dwh_hechos_estres_unificado');

CREATE TABLE dwh_kpi_resultados (
  id_resultado    INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_kpi          TINYINT UNSIGNED NOT NULL,
  codigo          VARCHAR(30)   NOT NULL,
  valor           DECIMAL(12,4) NOT NULL,
  detalle         VARCHAR(255)  NULL,
  fecha_ejecucion DATETIME      NOT NULL,
  PRIMARY KEY (id_resultado),
  CONSTRAINT fk_kpires_catalogo FOREIGN KEY (id_kpi) REFERENCES dwh_kpi_catalogo(id_kpi),
  INDEX idx_kpires_fecha (fecha_ejecucion)
) ENGINE=InnoDB;

-- ============================================================================
-- OBJETIVO 2: resultados de modelado (grano: dataset x modelo x ejecucion)
-- Comparte con la estrella la dimension dim_dataset (via id_dataset).
-- ============================================================================
CREATE TABLE dwh_modelo_resultados (
  id_resultado_modelo INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_dataset          TINYINT UNSIGNED NOT NULL COMMENT 'Dreaddit (train+test) se registra con id 8',
  tarea_modelado      VARCHAR(30)  NOT NULL,
  modelo              VARCHAR(60)  NOT NULL,
  protocolo           VARCHAR(25)  NOT NULL,
  n_registros         INT UNSIGNED NOT NULL,
  accuracy            DECIMAL(6,4) NOT NULL,
  f1                  DECIMAL(6,4) NOT NULL,
  auc                 DECIMAL(6,4) NOT NULL,
  auc_desv            DECIMAL(6,4) NULL COMMENT 'Solo en validacion cruzada',
  brecha_acc          DECIMAL(7,4) NOT NULL COMMENT 'Accuracy train - test (sobreajuste)',
  es_mejor_modelo     BOOLEAN NOT NULL,
  target              VARCHAR(80)  NOT NULL,
  lectura_capa_armonizada VARCHAR(160) NOT NULL,
  alerta              VARCHAR(160) NULL,
  fecha_ejecucion     DATETIME NOT NULL,
  PRIMARY KEY (id_resultado_modelo),
  CONSTRAINT fk_modres_dataset FOREIGN KEY (id_dataset) REFERENCES dim_dataset(id_dataset),
  INDEX idx_modres_ejecucion (fecha_ejecucion)
) ENGINE=InnoDB;

-- Vista de consumo: resultados de la ULTIMA ejecucion con la dimension dataset incorporada
CREATE OR REPLACE VIEW vw_modelo_resultados AS
SELECT r.id_resultado_modelo, r.id_dataset, ds.bloque, ds.dataset, ds.calidad, ds.uso_modelado,
       r.modelo, r.protocolo, r.n_registros, r.accuracy, r.f1, r.auc, r.auc_desv, r.brecha_acc,
       r.es_mejor_modelo, r.target, r.lectura_capa_armonizada, r.alerta, r.fecha_ejecucion
FROM dwh_modelo_resultados r
JOIN vw_dim_dataset ds ON ds.id_dataset = r.id_dataset
WHERE r.fecha_ejecucion = (SELECT MAX(fecha_ejecucion) FROM dwh_modelo_resultados);
"""
falta_dwh = [t for t in DWH_TABLAS if not existe(t)]
if falta_dwh:
    print("Faltan tablas del datawarehouse:", ", ".join(falta_dwh))
    print("-> se crea el datawarehouse completo (mismo contenido que dwh_estrella.sql; las tablas quedan vacías)...")
    sentencias = [s.strip() for s in "\n".join(l for l in DDL_DWH.splitlines() if not l.strip().startswith("--")).split(";") if s.strip()]
    with engine.begin() as con:
        for s in sentencias:
            con.execute(text(s))
    falta_dwh = [t for t in DWH_TABLAS if not existe(t)]
    if falta_dwh:
        raise RuntimeError("No se pudo crear el datawarehouse. Faltan: " + ", ".join(falta_dwh))
    print("Datawarehouse creado: 8 tablas y la vista vw_modelo_resultados.")
else:
    print("Datawarehouse: las 8 tablas existen.")

def nombres(tabla, id_col, nom_col):
    """Diccionario id -> nombre de una dimension (solo para mostrar tablas legibles)."""
    return dict(pd.read_sql(text(f"SELECT {id_col}, {nom_col} FROM {tabla}"), engine).itertuples(index=False))
GEN, RNG = nombres("dim_genero", "id_genero", "nombre"), nombres("dim_rango_edad", "id_rango_edad", "rango")
EDU, EST = nombres("dim_nivel_educativo", "id_nivel_educativo", "nivel"), nombres("dim_nivel_estres", "id_nivel_estres", "nivel")
BLQ, SUB = nombres("dim_bloque", "id_bloque", "nombre"), nombres("dim_subreddit", "id_subreddit", "nombre")

print(f"Conectado a {DB_NAME} | estrella: {n_estrella:,} registros | ejecución: {EJECUCION:%Y-%m-%d %H:%M:%S}")

### Celda 2 — Limpieza previa (se puede relanzar sin duplicar datos)
Las cinco tablas de transformación se vacían al empezar. Las tablas de KPIs **no**: `dwh_kpi_resultados` conserva el histórico (cada ejecución añade 6 filas con su fecha).

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
TABLAS_T = ["dwh_hechos_estres_unificado", "dwh_agg_prevalencia", "dwh_nlp_senales", "dwh_dass_severidad", "dwh_brecha_tratamiento"]
with engine.begin() as con:
    for t in TABLAS_T:
        n = con.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        con.execute(text(f"DELETE FROM {t}"))
        print(f"  {t:<30} {n:>8} filas previas eliminadas")
print("Tablas de transformación listas para una carga limpia.")

---
## T1 — Registros unificados (integración)
**Extrae** de la estrella los registros de las nueve fuentes con señal real (`uso_modelado = 'incluido'`; los tres datasets sintéticos quedan fuera, como acordó el tutor). **Transforma** añadiendo `demografia_completa` (género y edad reportados). **Carga** el hecho central del datawarehouse, que guarda solo claves hacia las dimensiones conformes.

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
q = """SELECT h.id_dataset, h.id_genero, h.id_rango_edad, h.id_nivel_educativo, h.id_nivel_estres
       FROM vw_hechos_estres h
       JOIN vw_dim_dataset ds ON ds.id_dataset = h.id_dataset
       WHERE ds.uso_modelado = 'incluido'"""
T1 = pd.read_sql(text(q), engine)
T1["demografia_completa"] = ((T1["id_genero"] != 0) & (T1["id_rango_edad"] != 0)).astype(int)
T1.to_sql("dwh_hechos_estres_unificado", engine, if_exists="append", index=False, chunksize=5000)
print(f"T1: {len(T1):,} registros -> dwh_hechos_estres_unificado")
bloques = pd.read_sql(text("SELECT d.id_dataset, b.nombre AS bloque FROM dim_dataset d JOIN dim_bloque b ON b.id_bloque = d.id_bloque"), engine)
display(T1.merge(bloques, on="id_dataset")["bloque"].value_counts().rename("registros").to_frame())

**Opinión.** La validación psicológica aporta el 94 % de los registros (301.095 de 319.318): da robustez estadística, pero inclina el volumen hacia población general; los bloques más estudiantiles suman solo el 6 %. Por eso cualquier lectura por bloque debe acompañarse de su tamaño muestral.

---
## T2 — Prevalencia de estrés (agregación)
Agrupa T1 por **bloque × género × rango de edad** y cuenta cuántos registros hay en cada nivel de estrés. Es un hecho agregado: Power BI lo consume sin recalcular y lo filtra con las mismas dimensiones que el resto.

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
ds_bloque = pd.read_sql(text("SELECT id_dataset, id_bloque FROM dim_dataset"), engine)
T2 = (T1.merge(ds_bloque, on="id_dataset")
        .groupby(["id_bloque", "id_genero", "id_rango_edad"])
        .agg(n_total=("id_nivel_estres", "size"),
             n_estres_alto=("id_nivel_estres", lambda s: int((s == 3).sum())),
             n_estres_moderado=("id_nivel_estres", lambda s: int((s == 2).sum())),
             n_estres_bajo=("id_nivel_estres", lambda s: int((s == 1).sum())))
        .reset_index())
T2["pct_estres_alto"] = (T2["n_estres_alto"] / T2["n_total"] * 100).round(2)
T2.to_sql("dwh_agg_prevalencia", engine, if_exists="append", index=False)
print(f"T2: {len(T2)} grupos -> dwh_agg_prevalencia")
vista = T2.assign(bloque=T2.id_bloque.map(BLQ), genero=T2.id_genero.map(GEN), rango_edad=T2.id_rango_edad.map(RNG))
display(vista.sort_values("pct_estres_alto", ascending=False)[["bloque", "genero", "rango_edad", "n_total", "pct_estres_alto"]].head(8))

**Opinión.** En los 12 grupos con más de 1.000 registros, el porcentaje de estrés Alto va del 14,7 % al 52,3 %: bloque, género y edad sí discriminan. Cuidado con los grupos pequeños: 18 de los 49 tienen menos de 30 registros y sus porcentajes extremos (100 %) no son interpretables. El dashboard debe mostrar siempre `n_total` junto al porcentaje.

---
## T3 — Señales lingüísticas de Dreaddit (resumen estadístico)
Resume los ~3.500 fragmentos de texto en una tabla compacta: por **subreddit y nivel de estrés** (Bajo = no estrés, Alto = estrés) calcula el sentimiento, las categorías LIWC de ansiedad y emociones, y la longitud media en palabras.

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
d = pd.read_sql(text("""SELECT id_subreddit, id_nivel_estres, sentimiento, liwc_ansiedad, liwc_emo_negativa, liwc_emo_positiva, texto
                        FROM b3_dreaddit_post"""), engine)
for c in ["sentimiento", "liwc_ansiedad", "liwc_emo_negativa", "liwc_emo_positiva"]:
    d[c] = d[c].astype(float)
d["n_words"] = d["texto"].str.split().str.len()
T3 = (d.groupby(["id_subreddit", "id_nivel_estres"])
        .agg(n_posts=("texto", "size"), sentimiento_medio=("sentimiento", "mean"),
             liwc_ansiedad_medio=("liwc_ansiedad", "mean"), liwc_emo_neg_medio=("liwc_emo_negativa", "mean"),
             liwc_emo_pos_medio=("liwc_emo_positiva", "mean"), longitud_media_palabras=("n_words", "mean"))
        .round(4).reset_index())
T3.to_sql("dwh_nlp_senales", engine, if_exists="append", index=False)
print(f"T3: {len(T3)} combinaciones subreddit x nivel -> dwh_nlp_senales")
display(T3.assign(subreddit=T3.id_subreddit.map(SUB), nivel=T3.id_nivel_estres.map(EST))
          [["subreddit", "nivel", "n_posts", "sentimiento_medio", "liwc_ansiedad_medio"]].head(8))

**Opinión.** En los 10 subreddits, los textos con estrés tienen más palabras de ansiedad (LIWC) y un sentimiento más bajo que los textos sin estrés de la misma comunidad: la señal no es un artefacto de un subreddit concreto. Esto avala usar las variables LIWC en el modelado de lenguaje del Objetivo 2.

---
## T4 — DASS por demografía con severidad clínica (pivote + umbrales)
Toma los respondentes de 18 a 25 años (proxy de población estudiantil) y calcula, por **género × nivel educativo**, las medias de las tres subescalas y el porcentaje en estrés **severo** (puntuación ≥ 40, baremo DASS).

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
d = pd.read_sql(text("""SELECT id_genero, id_rango_edad, id_nivel_educativo, score_depresion, score_ansiedad, score_estres
                        FROM b4_dass_respondente WHERE id_rango_edad = 2"""), engine)
T4 = (d.groupby(["id_genero", "id_rango_edad", "id_nivel_educativo"])
        .agg(n_respondentes=("score_estres", "size"), depresion_media=("score_depresion", "mean"),
             ansiedad_media=("score_ansiedad", "mean"), estres_medio=("score_estres", "mean"),
             pct_estres_severo=("score_estres", lambda s: (s >= 40).mean() * 100))
        .round(2).reset_index())
T4.to_sql("dwh_dass_severidad", engine, if_exists="append", index=False)
print(f"T4: {len(T4)} grupos demográficos -> dwh_dass_severidad")
display(T4.assign(genero=T4.id_genero.map(GEN), educacion=T4.id_nivel_educativo.map(EDU))
          .sort_values("estres_medio", ascending=False)
          [["genero", "educacion", "n_respondentes", "estres_medio", "pct_estres_severo"]].head(6))

**Opinión.** El grupo «Otro» con estudios de secundaria o menos concentra el malestar más alto (estrés medio 38,4 y 50 % en rango severo, n = 156), por encima de las mujeres con el mismo nivel educativo (36,5). Tanto en mujeres como en «Otro», el nivel de secundaria o menos supera al universitario. Nota: «Otro» incluye 37 respondentes de 18-25 años que no contestaron el género (código 0 del DASS), clasificados así por la ingesta.

---
## T5 — Brecha de tratamiento (tabla de contingencia)
Cruza, para los estudiantes del dataset de salud mental, el **estrés creciente** (Bajo / Moderado / Alto) con los **antecedentes familiares** y cuenta cuántos han buscado tratamiento. Es una contingencia de dos dimensiones: una conformada (nivel de estrés) y otra degenerada (antecedentes).

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
d = pd.read_sql(text("""SELECT id_nivel_estres, antecedentes_familiares, tratamiento FROM b4_mental_health
                        WHERE ocupacion = 'Student' AND antecedentes_familiares IS NOT NULL AND tratamiento IS NOT NULL"""), engine)
T5 = (d.groupby(["id_nivel_estres", "antecedentes_familiares"])
        .agg(n_total=("tratamiento", "size"), n_con_tratamiento=("tratamiento", lambda s: int((s == "Yes").sum())))
        .reset_index())
T5["pct_tratamiento"] = (T5["n_con_tratamiento"] / T5["n_total"] * 100).round(2)
T5.to_sql("dwh_brecha_tratamiento", engine, if_exists="append", index=False)
print(f"T5: {len(T5)} celdas de la contingencia -> dwh_brecha_tratamiento")
display(T5.assign(estres_creciente=T5.id_nivel_estres.map(EST))[["estres_creciente", "antecedentes_familiares", "n_total", "pct_tratamiento"]])
por_estres = T5.groupby("id_nivel_estres")[["n_total", "n_con_tratamiento"]].sum()
por_antec = T5.groupby("antecedentes_familiares")[["n_total", "n_con_tratamiento"]].sum()
print("\n% con tratamiento según estrés creciente:  ", {EST[i]: round(r.n_con_tratamiento / r.n_total * 100, 1) for i, r in por_estres.iterrows()})
print("% con tratamiento según antecedentes:       ", {i: round(r.n_con_tratamiento / r.n_total * 100, 1) for i, r in por_antec.iterrows()})

**Opinión.** La contingencia es contundente: dentro de cada nivel de estrés, los antecedentes familiares marcan la diferencia (≈ 35 % sin antecedentes frente a ≈ 72-73 % con ellos), mientras que el nivel de estrés apenas mueve el resultado (48-51 %). Es la brecha entre malestar percibido y búsqueda de ayuda: la familiaridad con la salud mental pesa más que la intensidad del malestar.

---
## KPIs y almacenamiento con histórico
Seis indicadores calculados con Python sobre las tablas anteriores y las ingestadas. Cada ejecución inserta 6 filas en `dwh_kpi_resultados` con su fecha, de modo que el valor de un KPI nunca se sobrescribe. La definición de cada uno está en `dwh_kpi_catalogo` y en el documento `KPIs_TFM.docx`.

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
if "T5" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T5. Usa Kernel > Restart Kernel and Run All Cells.")
kpis = []
# KPI 1 - Tasa de estrés Alto (sobre T1)
n_alto = int((T1["id_nivel_estres"] == 3).sum())
kpis.append((1, "KPI_ESTRES_ALTO", round(n_alto / len(T1) * 100, 4), f"{n_alto:,} de {len(T1):,} registros"))
# KPI 2 - Brecha de género en ansiedad, DASS 18-25 (mujer - hombre)
a = pd.read_sql(text("SELECT id_genero, AVG(score_ansiedad) AS a FROM b4_dass_respondente WHERE id_rango_edad = 2 AND id_genero IN (1, 2) GROUP BY id_genero"),
                engine).set_index("id_genero")["a"].astype(float)
kpis.append((2, "KPI_BRECHA_GENERO", round(a[2] - a[1], 4), f"Mujer {a[2]:.1f} - Hombre {a[1]:.1f}"))
# KPI 3 - Riesgo crítico: % de estudiantes con pensamientos suicidas que presentan depresión
k3 = float(pd.read_sql(text("SELECT 100.0 * AVG(CASE WHEN depresion = 'Yes' THEN 1.0 ELSE 0.0 END) AS v FROM b3_depression_student WHERE pensamientos_suicidas = 'Yes'"), engine).iloc[0, 0])
kpis.append((3, "KPI_RIESGO_CRITICO", round(k3, 4), "% de estudiantes con pensamientos suicidas que presentan depresión"))
# KPI 4 - Tasa de búsqueda de tratamiento (sobre T5)
kpis.append((4, "KPI_TASA_TRATAMIENTO", round(T5["n_con_tratamiento"].sum() / T5["n_total"].sum() * 100, 4), f"{int(T5['n_con_tratamiento'].sum()):,} de {int(T5['n_total'].sum()):,} estudiantes"))
# KPI 5 - Poder discriminante LIWC: ansiedad en estrés / ansiedad en no estrés
lw = pd.read_sql(text("SELECT etiqueta_estres, AVG(liwc_ansiedad) AS a FROM b3_dreaddit_post GROUP BY etiqueta_estres"), engine).set_index("etiqueta_estres")["a"].astype(float)
kpis.append((5, "KPI_DISCRIMINANTE_LIWC", round(lw[1] / lw[0], 4), f"estrés {lw[1]:.2f} / no estrés {lw[0]:.2f}"))
# KPI 6 - Cobertura demográfica (sobre T1)
kpis.append((6, "KPI_COBERTURA_DEMO", round(T1["demografia_completa"].mean() * 100, 4), "% de registros con género y edad reportados"))

res = pd.DataFrame(kpis, columns=["id_kpi", "codigo", "valor", "detalle"])
res["fecha_ejecucion"] = EJECUCION
res.to_sql("dwh_kpi_resultados", engine, if_exists="append", index=False)
print("KPIs calculados y almacenados:")
display(res[["codigo", "valor", "detalle"]])

---
## Verificación final
Tres comprobaciones sobre la base de datos:
1. **KPIs** de la última ejecución, uniendo catálogo y resultados (es la consulta que alimentará las tarjetas de Power BI).
2. **Integridad de la estrella:** ninguna clave foránea huérfana en ninguna tabla de hechos.
3. **Cuadre con la estrella:** los registros 'incluido' de `vw_hechos_estres` deben sumar lo mismo que T1.

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
if "res" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define res. Usa Kernel > Restart Kernel and Run All Cells.")
q = """SELECT c.codigo, c.nombre, r.valor, c.unidad, r.detalle
       FROM dwh_kpi_resultados r JOIN dwh_kpi_catalogo c ON c.id_kpi = r.id_kpi
       WHERE r.fecha_ejecucion = (SELECT MAX(fecha_ejecucion) FROM dwh_kpi_resultados) ORDER BY c.id_kpi"""
display(pd.read_sql(text(q), engine))

CLAVES = [("dwh_hechos_estres_unificado", "id_dataset", "dim_dataset"), ("dwh_hechos_estres_unificado", "id_genero", "dim_genero"),
          ("dwh_hechos_estres_unificado", "id_rango_edad", "dim_rango_edad"), ("dwh_hechos_estres_unificado", "id_nivel_educativo", "dim_nivel_educativo"),
          ("dwh_hechos_estres_unificado", "id_nivel_estres", "dim_nivel_estres"), ("dwh_agg_prevalencia", "id_bloque", "dim_bloque"),
          ("dwh_agg_prevalencia", "id_genero", "dim_genero"), ("dwh_agg_prevalencia", "id_rango_edad", "dim_rango_edad"),
          ("dwh_nlp_senales", "id_subreddit", "dim_subreddit"), ("dwh_nlp_senales", "id_nivel_estres", "dim_nivel_estres"),
          ("dwh_dass_severidad", "id_genero", "dim_genero"), ("dwh_dass_severidad", "id_rango_edad", "dim_rango_edad"),
          ("dwh_dass_severidad", "id_nivel_educativo", "dim_nivel_educativo"), ("dwh_brecha_tratamiento", "id_nivel_estres", "dim_nivel_estres")]
filas = []
with engine.connect() as con:
    for t, col, dim in CLAVES:
        huerfanas = con.execute(text(f"SELECT COUNT(*) FROM {t} f LEFT JOIN {dim} x ON x.{col} = f.{col} WHERE x.{col} IS NULL")).scalar()
        total = con.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        filas.append({"tabla de hechos": t, "clave": col, "dimensión": dim, "filas": total, "huérfanas": huerfanas})
integ = pd.DataFrame(filas)
display(integ)
print("INTEGRIDAD OK: 0 claves huérfanas." if integ["huérfanas"].sum() == 0 else "ATENCIÓN: hay claves huérfanas.")

with engine.connect() as con:
    tot, reales = con.execute(text("""SELECT COUNT(*), SUM(CASE WHEN ds.uso_modelado = 'incluido' THEN 1 ELSE 0 END)
                                      FROM vw_hechos_estres h JOIN vw_dim_dataset ds ON ds.id_dataset = h.id_dataset""")).one()
    en_dwh = con.execute(text("SELECT COUNT(*) FROM dwh_hechos_estres_unificado")).scalar()
print(f"\nEstrella: {int(tot):,} registros ({int(reales):,} de fuentes con señal real) | T1 en el DWH: {en_dwh:,}")
print("CUADRE OK: el DWH coincide con las fuentes 'incluido' de la estrella." if int(reales) == en_dwh else "ATENCIÓN: no cuadra; revisa que el notebook 07 se ejecutó una sola vez.")
print("\n✅ ETL ENTREGA 2 COMPLETADO — notebook 08 v4")